# Phase 1 — Dataset Preprocessing

## Goal
Load the XSum dataset, explore its structure, clean the text, split it into train/test sets, and save it for the next phase.

## Dataset
**XSum** — BBC news articles with one-sentence human-written summaries.
- `document` → full news article (model input)
- `summary` → one-sentence summary (target label)

In [12]:
import os
os.environ["HF_HOME"] = r"D:\hf_cache"

from datasets import load_dataset
import numpy as np
import re

## 1. Load Dataset

In [13]:
print("Loading dataset...")
dataset = load_dataset("xsum", split="train[:2000]")

print(f"Total examples: {len(dataset)}")
print(dataset)

Loading dataset...
Total examples: 2000
Dataset({
    features: ['document', 'summary', 'id'],
    num_rows: 2000
})


## 2. Explore Dataset (Before Cleaning)

In [14]:
print("Document (input):")
print(dataset[0]["document"][:300])
print("\nSummary (target):")
print(dataset[0]["summary"][:200])

Document (input):
The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.
Trains on the west coast mainline face disruption due to damage at the Lamington Viaduct.
Many

Summary (target):
Clean-up operations are continuing across the Scottish Borders and Dumfries and Galloway after flooding caused by Storm Frank.


In [15]:
doc_lengths = [len(x["document"].split()) for x in dataset]
sum_lengths = [len(x["summary"].split())  for x in dataset]

print("--- Length statistics (before cleaning) ---")
print(f"Avg document length: {np.mean(doc_lengths):.0f} words")
print(f"Avg summary length:  {np.mean(sum_lengths):.0f} words")
print(f"Max document length: {max(doc_lengths)} words")
print(f"Min document length: {min(doc_lengths)} words")

--- Length statistics (before cleaning) ---
Avg document length: 375 words
Avg summary length:  21 words
Max document length: 2694 words
Min document length: 11 words


## 3. Clean Text
Remove noise: reference numbers, LaTeX math, URLs, extra whitespace.
We do NOT apply stemming or stopword removal because BART relies on full contextual understanding.

In [16]:
def clean_text(text):
    text = re.sub(r'\[\d+\]', '', text)   # Remove [1], [23]
    text = re.sub(r'\$.*?\$', '', text)   # Remove LaTeX $x^2$
    text = re.sub(r'http\S+', '', text)   # Remove URLs
    text = re.sub(r'\s+', ' ', text)      # Collapse whitespace
    text = text.strip()
    return text

def preprocess_sample(sample):
    return {
        "document": clean_text(sample["document"]),
        "summary":  clean_text(sample["summary"])
    }

dataset = dataset.map(preprocess_sample)
print("Cleaning done!")

Cleaning done!


## 4. Length Statistics (After Cleaning)

In [17]:
doc_lengths_clean = [len(x["document"].split()) for x in dataset]
sum_lengths_clean = [len(x["summary"].split())  for x in dataset]

print("--- Length statistics (after cleaning) ---")
print(f"Avg document length: {np.mean(doc_lengths_clean):.0f} words")
print(f"Avg summary length:  {np.mean(sum_lengths_clean):.0f} words")

--- Length statistics (after cleaning) ---
Avg document length: 375 words
Avg summary length:  21 words


## 5. Split & Save
80% training, 20% testing. `seed=42` ensures the same split every run.

In [18]:
split = dataset.train_test_split(test_size=0.2, seed=42)
train_data = split["train"]
test_data  = split["test"]

print(f"Training examples: {len(train_data)}")
print(f"Testing examples:  {len(test_data)}")

os.makedirs("../data/processed", exist_ok=True)
train_data.save_to_disk("../data/processed/train")
test_data.save_to_disk("../data/processed/test")

print("Dataset saved!")
print("Phase 1 complete!")

Training examples: 1600
Testing examples:  400


Saving the dataset (1/1 shards): 100%|██████████| 400/400 [00:00<00:00, 68411.42 examples/s]

Dataset saved!
Phase 1 complete!
